In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.signal import find_peaks

# =================================================
# USER SETTINGS
# =================================================
BASE_DIR = os.getcwd()   # Run script from Trieste directory
OUTPUT_EXCEL = "Trieste_Error_Metrics_Peaks99_All_Emulators.xlsx"
TEST_YEARS = [1995, 1996, 2020]
PERC = np.arange(0, 101, 1)
PEAK_PERCENTILE = 99  # threshold for extremes

# =================================================
# HELPER FUNCTIONS
# =================================================
def extract_run_number(filename):
    """
    Extract run number from filenames like:
    lstmHybridModel_Run12_testresults.txt
    """
    return int(filename.split("Run")[1].split("_")[0])


def compute_metrics_peaks(file_path):
    """
    Compute performance metrics for storm surge peaks
    EXACTLY as in the original single-run script.
    """
    dyn = np.loadtxt(file_path)

    # Filter test years
    idx_test = np.isin(dyn[:, 0], TEST_YEARS)
    dynTest = dyn[idx_test]

    # Remove mean
    xx = dynTest[:, 8] - np.nanmean(dynTest[:, 8])  # Observed
    yy = dynTest[:, 7] - np.nanmean(dynTest[:, 7])  # Emulator

    # -------------------------------------------------
    # Identify peaks ABOVE the 99th percentile
    # (same logic as original code)
    # -------------------------------------------------
    storm_surge_threshold = np.percentile(xx, PEAK_PERCENTILE)
    peaks, _ = find_peaks(xx, height=storm_surge_threshold)

    if len(peaks) < 2:
        return None

    # Extract peaks
    x = xx[peaks]
    y = yy[peaks]

    # Remove NaNs
    valid = ~np.isnan(x)
    xr = x[valid]
    yr = y[valid]

    if len(xr) < 2:
        return None

    # -------------------------------------------------
    # Linear regression
    # -------------------------------------------------
    X = sm.add_constant(xr)
    model = sm.OLS(yr, X).fit()
    slope = model.params[1]

    # -------------------------------------------------
    # Metrics (IDENTICAL to original)
    # -------------------------------------------------
    slf = slope
    corr = np.corrcoef(xr, yr)[0, 1]
    rmse = np.sqrt(np.mean((xr - yr) ** 2))
    bias = np.sum(yr - xr) / len(xr)
    mad = np.nanmean(np.abs(xr - yr))

    sta_prc = np.percentile(xr, np.arange(0, 101, 1))
    em_prc  = np.percentile(yr, np.arange(0, 101, 1))

    madp = np.nanmean(np.abs(em_prc - sta_prc))
    madc = mad + madp

    return slf, corr, rmse, bias, mad, madp, madc

# =================================================
# MAIN PROCESSING LOOP
# =================================================
writer = pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl")

for emulator_folder in sorted(os.listdir(BASE_DIR)):

    folder_path = os.path.join(BASE_DIR, emulator_folder)

    # Skip non-directories
    if not os.path.isdir(folder_path):
        continue

    # Collect test result files
    run_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".txt") and "testresults" in f
    ]

    if len(run_files) == 0:
        continue

    # Sort runs numerically
    run_files = sorted(run_files, key=extract_run_number)

    print(f"Processing {emulator_folder} ({len(run_files)} runs)")

    results = []
    run_ids = []

    for run_file in run_files:
        file_path = os.path.join(folder_path, run_file)
        metrics = compute_metrics_peaks(file_path)

        if metrics is not None:
            results.append(metrics)
            run_ids.append(extract_run_number(run_file))
        else:
            print(f"  Skipping {run_file}: insufficient peak data")

    if len(results) == 0:
        print(f"  No valid peak results for {emulator_folder}")
        continue

    # Create DataFrame
    df = pd.DataFrame(
        results,
        columns=["SLF", "Corr", "RMSE", "Bias", "MAD", "MADp", "MADc"],
        index=run_ids
    )
    df.index.name = "Run"

    # Excel sheet name (max 31 chars)
    sheet_name = emulator_folder[:31]
    df.to_excel(writer, sheet_name=sheet_name, float_format="%.4f")

# Save Excel file
writer.close()

print(f"\nAll peak-based results saved to: {OUTPUT_EXCEL}")


Processing LSTM-MADc (40 runs)
Processing LSTM-MSE (40 runs)
Processing LSTMh-MADc (40 runs)
Processing LSTMh-MSE (40 runs)
Processing MLP-MADc (40 runs)
Processing MLP-MSE (40 runs)
Processing MLR-MADc (40 runs)
Processing MLR-MSE (40 runs)
Processing RNN-MADc (40 runs)
Processing RNN-MSE (40 runs)
Processing RNNh-MADc (40 runs)
Processing RNNh-MSE (40 runs)

All peak-based results saved to: Trieste_Error_Metrics_Peaks99_All_Emulators.xlsx
